# Content Based Recommendation System

In [145]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

## Data Collection

In [3]:
books = pd.read_csv('Books.csv')
books.head()

,bookId,title,series,author,rating,description,language,isbn,genres,characters,...,firstPublishDate,awards,numRatings,ratingsByStars,likedPercent,setting,coverImg,bbeScore,bbeVotes,price
0,2767052-the-hunger-games,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,English,9780439023481,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...","['Katniss Everdeen', 'Peeta Mellark', 'Cato (H...",...,NaN,['Locus Award Nominee for Best Young Adult Boo...,6376780,"['3444695', '1921313', '745221', '171994', '93...",96.0,"['District 12, Panem', 'Capitol, Panem', 'Pane...",https://i.gr-assets.com/images/S/compressed.ph...,2993816,30516,5.09
1,2.Harry_Potter_and_the_Order_of_the_Phoenix,Harry Potter and the Order of the Phoenix,Harry Potter #5,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,There is a door at the end of a silent corrido...,English,9780439358071,"['Fantasy', 'Young Adult', 'Fiction', 'Magic',...","['Sirius Black', 'Draco Malfoy', 'Ron Weasley'...",...,06/21/03,['Bram Stoker Award for Works for Young Reader...,2507623,"['1593642', '637516', '222366', '39573', '14526']",98.0,['Hogwarts School of Witchcraft and Wizardry (...,https://i.gr-assets.com/images/S/compressed.ph...,2632233,26923,7.38
2,2657.To_Kill_a_Mockingbird,To Kill a Mockingbird,To Kill a Mockingbird,Harper Lee,4.28,The unforgettable novel of a childhood in a sl...,English,9999999999999,"['Classics', 'Fiction', 'Historical Fiction', ...","['Scout Finch', 'Atticus Finch', 'Jem Finch', ...",...,07/11/60,"['Pulitzer Prize for Fiction (1961)', 'Audie A...",4501075,"['2363896', '1333153', '573280', '149952', '80...",95.0,"['Maycomb, Alabama (United States)']",https://i.gr-assets.com/images/S/compressed.ph...,2269402,23328,NaN
3,1885.Pride_and_Prejudice,Pride and Prejudice,NaN,"Jane Austen, Anna Quindlen (Introduction)",4.26,Alternate cover edition of ISBN 9780679783268S...,English,9999999999999,"['Classics', 'Fiction', 'Romance', 'Historical...","['Mr. Bennet', 'Mrs. Bennet', 'Jane Bennet', '...",...,01/28/13,[],2998241,"['1617567', '816659', '373311', '113934', '767...",94.0,"['United Kingdom', 'Derbyshire, England (Unite...",https://i.gr-assets.com/images/S/compressed.ph...,1983116,20452,NaN
4,41865.Twilight,Twilight,The Twilight Saga #1,Stephenie Meyer,3.60,About three things I was absolutely positive.\...,English,9780316015844,"['Young Adult', 'Fantasy', 'Romance', 'Vampire...","['Edward Cullen', 'Jacob Black', 'Laurent', 'R...",...,10/05/05,"['Georgia Peach Book Award (2007)', 'Buxtehude...",4964519,"['1751460', '1113682', '1008686', '542017', '5...",78.0,"['Forks, Washington (United States)', 'Phoenix...",https://i.gr-assets.com/images/S/compressed.ph...,1459448,14874,2.1


In [4]:
books.shape

(52478, 25)

In [5]:
books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52478 entries, 0 to 52477
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   bookId            52478 non-null  object 
 1   title             52478 non-null  object 
 2   series            23470 non-null  object 
 3   author            52478 non-null  object 
 4   rating            52478 non-null  float64
 5   description       51140 non-null  object 
 6   language          48672 non-null  object 
 7   isbn              52478 non-null  object 
 8   genres            52478 non-null  object 
 9   characters        52478 non-null  object 
 10  bookFormat        51005 non-null  object 
 11  edition           4955 non-null   object 
 12  pages             50131 non-null  object 
 13  publisher         48782 non-null  object 
 14  publishDate       51598 non-null  object 
 15  firstPublishDate  31152 non-null  object 
 16  awards            52478 non-null  object

## Data Preparation

### 1. Remove unnecessary columns, handle missing values and clean data

In [8]:
#Finding number of missing values in books dataframe
print("Missing values per column:")
print(books.isnull().sum())

print("\nRows with missing values:")
print(books[books.isnull().any(axis=1)])

Missing values per column:
bookId                  0
title                   0
series              29008
author                  0
rating                  0
description          1338
language             3806
isbn                    0
genres                  0
characters              0
bookFormat           1473
edition             47523
pages                2347
publisher            3696
publishDate           880
firstPublishDate    21326
awards                  0
numRatings              0
ratingsByStars          0
likedPercent          622
setting                 0
coverImg              605
bbeScore                0
bbeVotes                0
price               14365
dtype: int64

Rows with missing values:
                             bookId                  title  \
0          2767052-the-hunger-games       The Hunger Games   
2        2657.To_Kill_a_Mockingbird  To Kill a Mockingbird   
3          1885.Pride_and_Prejudice    Pride and Prejudice   
4                    41865.Twilight

There are a lot of missing data in the dataset. Each column is to be handled  separately. Add missingflags for columns retained
1. series - replace NaN with "Not specified"
2. description - Genre field does not have any missing values. So, delete Description field.
3. language - replace NaN with "Not specified"
4. bookFormat - delete
5. edition - replace NaN with "Not specified"
6. pages - replace nan with median
7. publisher - replace NaN with "Not specified"
8. publishDate - delete
9. firstPublishDate - Median of author publish years - then for additional nans,replace with common median
10. likedPercent - replace with median
11. coverImage - delete
12. price - replace by  median

In [10]:
#Delete publishDate and coverImage Columns
books.drop(['publishDate', 'coverImg'], axis=1, inplace=True)

In [11]:
#Dropping bookFormat field
books.drop('bookFormat', axis=1, inplace=True)

In [12]:
#Dropping description field
books.drop('description', axis=1, inplace=True)

#### Adding missing flag (a column with 1 for row with missing value and 0 otherwise) for the columns with missing values

In [14]:
for col in books.columns:
    if books[col].isna().any():
        #Add missing flag
        books[col+'_missing'] = books[col].isna().astype(int)

In [15]:
books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52478 entries, 0 to 52477
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   bookId                    52478 non-null  object 
 1   title                     52478 non-null  object 
 2   series                    23470 non-null  object 
 3   author                    52478 non-null  object 
 4   rating                    52478 non-null  float64
 5   language                  48672 non-null  object 
 6   isbn                      52478 non-null  object 
 7   genres                    52478 non-null  object 
 8   characters                52478 non-null  object 
 9   edition                   4955 non-null   object 
 10  pages                     50131 non-null  object 
 11  publisher                 48782 non-null  object 
 12  firstPublishDate          31152 non-null  object 
 13  awards                    52478 non-null  object 
 14  numRat

In [16]:
#series - replace NaN with "Not specified"
books['series'] = books['series'].fillna("not specified")

In [17]:
#language - replace NaN with "Not specified"
books['language'] = books['language'].fillna("not specified")

In [18]:
#edition - replace NaN with "Not specified"
books['edition'] = books['edition'].fillna("not specified")

In [19]:
#publisher - replace NaN with "Not specified"
books['publisher'] = books['publisher'].fillna("not specified")

In [20]:
#Replacing pages with median value
#The datatype of pages column is object (Refer books.info()). It need to be converted to numeric and then replace nan with median
# Step 1: convert to numeric
books['pages'] = pd.to_numeric(books['pages'], errors='coerce')
# Step 2: fill NaNs
books['pages'] = books['pages'].fillna(books['pages'].median())
# Step 3: convert to int
books['pages'] = books['pages'].astype(int)

In [21]:
#likedPercent - replace with median
books['likedPercent'] = books['likedPercent'].fillna(books['likedPercent'].median())

In [22]:
#firstPublishDate - Median of author publish years - then for additional nans,replace with common median
#step1 - convert to datetime and extract the year
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='pandas')
books['firstPublishDate'] = pd.to_datetime(books['firstPublishDate'], errors='coerce')
#Step2 - extract year and store in the column
books['firstPublishDate'] = books['firstPublishDate'].dt.year
#Step3 - replace nan with authorwise median
books['firstPublishDate'] = books.groupby('author')['firstPublishDate'] \
                                 .transform(lambda x: x.fillna(x.median()))
#Step 4- replace remaining nans with global median
books['firstPublishDate'] = books['firstPublishDate'].fillna(books['firstPublishDate'].median())
#Step5 - convert to int
books['firstPublishDate'] = books['firstPublishDate'].astype(int)

C:\Users\Anisha\AppData\Local\Temp\ipykernel_30816\1427318583.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  books['firstPublishDate'] = pd.to_datetime(books['firstPublishDate'], errors='coerce')


In [23]:
#price - replace by  median and convert to float
books['price'] = pd.to_numeric(books['price'], errors='coerce')
books['price'] = books['price'].fillna(books['price'].median())
books['price'] = books['price'].astype(float)

In [24]:
#Replace years over current year with median
from datetime import datetime
current_year = int(datetime.now().year)
books.loc[books['firstPublishDate'] > current_year, 'firstPublishDate'] = books['firstPublishDate'].median()

### 2. Data encoding and Feature Engineering

In [26]:
books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52478 entries, 0 to 52477
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   bookId                    52478 non-null  object 
 1   title                     52478 non-null  object 
 2   series                    52478 non-null  object 
 3   author                    52478 non-null  object 
 4   rating                    52478 non-null  float64
 5   language                  52478 non-null  object 
 6   isbn                      52478 non-null  object 
 7   genres                    52478 non-null  object 
 8   characters                52478 non-null  object 
 9   edition                   52478 non-null  object 
 10  pages                     52478 non-null  int32  
 11  publisher                 52478 non-null  object 
 12  firstPublishDate          52478 non-null  int32  
 13  awards                    52478 non-null  object 
 14  numRat

From above, we can see that there are many non numeric fields. For training, we need to convert them.

In [28]:
#Delete bookId column as isbn column uniquely identifies the book
books.drop('bookId', axis=1, inplace=True)

In [91]:
# Get all columns with dtype 'object'
object_cols = books.select_dtypes(include='object').columns.tolist()

In [30]:
#Check contents of isbn.
books['isbn'].unique()

array(['9780439023481', '9780439358071', '9999999999999', ...,
       '9781461017097', '9781450755634', '9781599554976'], dtype=object)

In [31]:
#Convert isbn to int64
# Step 1: Remove non-digit characters (hyphens, spaces, etc.)
books['isbn'] = books['isbn'].str.replace(r'\D', '', regex=True)

# Step 2: Convert to numeric safely with int64
books['isbn'] = pd.to_numeric(books['isbn'], errors='coerce')

In [32]:
#Remove isbn from object_cols
object_cols.remove('isbn')

In [33]:
#Finding number of unique values in each object column
for col in object_cols:
    print(col+' '+str(books[col].nunique()))

title 49927
series 22803
author 28227
language 82
genres 44154
characters 12448
edition 1789
publisher 11111
awards 9215
ratingsByStars 49908
setting 4651


#### Dealing with ratingByStars

In [35]:
#each ratingByStars value is a list of 5 numbers, where value[x] is the number users who has given x+1 stars
#the field numRatings give the total number of user ratings
print(books.loc[0,'ratingsByStars'])

['3444695', '1921313', '745221', '171994', '93557']


In [37]:
#Split the ratingsByStars field into 5 fields star5,star4,star3,star2 and star1 with % of users chosen that star
# Convert each string in the list to int

import ast
# Convert string representation to list
books['ratingsByStars'] = books['ratingsByStars'].apply(ast.literal_eval)

# Convert each element to int
books['ratingsByStars'] = books['ratingsByStars'].apply(lambda lst: [int(x) for x in lst])

# Calculate percentage for each element in the list - the lists may have different lengths. Filling extra with zeros
max_len = books['ratingsByStars'].apply(len).max()

for i in range(max_len):
    books[f'star{i+1}'] = books.apply(
        lambda row: (int(row['ratingsByStars'][i]) / row['numRatings'] * 100) if i < len(row['ratingsByStars']) else 0,
        axis=1
    )
    
#Delete numRatings and ratingsByStars
books.drop(['numRatings','ratingsByStars'], axis=1, inplace=True)
#Delete ratingsByStars from object_cols
object_cols.remove('ratingsByStars')



In [39]:
books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52478 entries, 0 to 52477
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   title                     52478 non-null  object 
 1   series                    52478 non-null  object 
 2   author                    52478 non-null  object 
 3   rating                    52478 non-null  float64
 4   language                  52478 non-null  object 
 5   isbn                      52477 non-null  float64
 6   genres                    52478 non-null  object 
 7   characters                52478 non-null  object 
 8   edition                   52478 non-null  object 
 9   pages                     52478 non-null  int32  
 10  publisher                 52478 non-null  object 
 11  firstPublishDate          52478 non-null  int32  
 12  awards                    52478 non-null  object 
 13  likedPercent              52478 non-null  float64
 14  settin

genre and settings columns have list of values in each row. They are to be considered individually and converted into multi hot encoding and then delete from object_cols

In [73]:
#Create a genre vocabulary for multi-hot encoding - separate each genre
all_genres = sorted(
    set(genre for genres_list in books["genres"] for genre in genres_list)
)

genre_to_index = {genre: i for i, genre in enumerate(all_genres)}

#Create a settings vocabulary for multi-hot encoding - separate each setting
all_setting = sorted(
    set(setting for setting_list in books["setting"] for setting in setting_list)
)

setting_to_index = {setting: i for i, setting in enumerate(all_setting)}

In [75]:
#Function to create multi hot encoding vector
def create_multi_hot(values_list, vocab, vocab_index):
    vec = np.zeros(len(vocab))
    for v in values_list:
        if v in vocab_index:
            vec[vocab_index[v]] = 1
    return vec

In [93]:
#Delete genres and setting from object_cols
object_cols.remove('genres')
object_cols.remove('setting')
object_cols

['title',
 'series',
 'author',
 'language',
 'characters',
 'edition',
 'publisher',
 'awards']

In [56]:
#Store numerical columns in a list
# Get all columns with dtype 'object'
num_cols = books.select_dtypes(include='number').columns.tolist()

print(num_cols)

['rating', 'isbn', 'pages', 'firstPublishDate', 'likedPercent', 'bbeScore', 'bbeVotes', 'price', 'series_missing', 'language_missing', 'edition_missing', 'pages_missing', 'publisher_missing', 'firstPublishDate_missing', 'likedPercent_missing', 'price_missing', 'star1', 'star2', 'star3', 'star4', 'star5']


##### The ultimate aim is to find an encoding for each row in the bookset. Here, we combine the encodings for all the numeric and character columns to form a single vector. 

First we need to encode the columns in object_cols. For that we are using sentence_transformers library which is build on BERT. Loads a pretrained BERT-based model.'all-MiniLM-L6-v2' is a lightweight sentence-embedding model.It converts sentences into dense numeric vectors (embeddings). Then, normalize numeric columns. Also, multi hot encode genres and setting fields. Finally combine them all together to form a vector per row

In [95]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
model = SentenceTransformer('all-MiniLM-L6-v2')
# ---- STEP 1: Create embeddings for object columns ----
for col in object_cols:
    col_data = books[col].fillna("").astype(str).tolist()
    embeddings = model.encode(col_data)
    books[col + "_embeddings"] = list(embeddings)


# ---- STEP 2: Normalize numeric columns ----
numeric_features = books[num_cols] / books[num_cols].max()
numeric_features = numeric_features.to_numpy()


# ---- STEP 3: Combine all features ----
combined_features = []

for i in range(len(books)):

    # Multi-hot genre and setting
    genre_vec = create_multi_hot(books.iloc[i]["genres"], all_genres, genre_to_index)
    setting_vec = create_multi_hot(books.iloc[i]["setting"], all_setting, setting_to_index)

    # Collect embeddings from object columns
    object_fields_combined = []
    for col in object_cols:
        embedding_vector = books.iloc[i][col + "_embeddings"]
        object_fields_combined.append(embedding_vector)

    # Flatten embedding vectors
    object_fields_combined = np.concatenate(object_fields_combined)

    # Final combined vector
    combined_vec = np.concatenate([
        object_fields_combined,
        genre_vec,
        setting_vec,
        numeric_features[i]
    ])

    combined_features.append(combined_vec)

combined_features = np.array(combined_features)

C:\Users\Anisha\anaconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## Recommendation function

In [117]:
#Function to recommend similar books using cosine similarity
import tensorflow as tf
import numpy as np

def recommend_combined_tf(book_title, top_k=2):
    # Get index of the book
    matches = books.index[books["title"] == book_title].tolist()
    if not matches:
        return []  # return empty list
    
    idx = matches[0]
    
    # Convert combined_features to a TensorFlow tensor
    features_tensor = tf.convert_to_tensor(combined_features, dtype=tf.float32)
    
    # Compute cosine similarity
    # cosine_sim = (A · B) / (||A|| * ||B||)
    target = tf.expand_dims(features_tensor[idx], axis=0)  # shape (1, feature_dim)
    dot_product = tf.linalg.matmul(target, features_tensor, transpose_b=True)  # shape (1, num_books)
    
    # Norms
    target_norm = tf.norm(target, axis=1, keepdims=True)
    all_norms = tf.norm(features_tensor, axis=1, keepdims=True)
    similarity = dot_product / (target_norm * tf.transpose(all_norms))  # shape (1, num_books)
    
    similarity = tf.squeeze(similarity)  # shape (num_books,)
    
    # Get top indices excluding the book itself
    top_indices = tf.argsort(similarity, direction='DESCENDING')
    top_indices = [i.numpy() for i in top_indices if i != idx]  # remove the book itself
    

    # Handle fewer books than top_k
    top_indices = top_indices[:min(top_k, len(top_indices))]
    
    # Prepare recommendations
    recommendations = [
        (books.iloc[i]["title"], float(similarity[i].numpy()))
        for i in top_indices
    ]
    
    return recommendations

In [119]:
# Example recommendation
book = "The Hunger Games"
recs = recommend_combined_tf(book, top_k=2)
print(f"Since you have chosen {book} here are some recommendations...")
for title, score in recs:
    print(f"{title} (similarity: {score:.2f})")

Since you have chosen The Hunger Games here are some recommendations...
Catching Fire (similarity: 0.97)
Mockingjay (similarity: 0.96)


In [121]:
#Example
book = "Harry Potter and the Order of the Phoenix"
recs = recommend_combined_tf(book, top_k=10)
print(f"Since you have chosen {book} here are some recommendations...")
for title, score in recs:
    print(f"{title} (similarity: {score:.2f})")

Since you have chosen Harry Potter and the Order of the Phoenix here are some recommendations...
Harry Potter and the Sorcerer's Stone (similarity: 0.98)
Harry Potter and the Prisoner of Azkaban (similarity: 0.97)
Harry Potter and the Goblet of Fire (similarity: 0.97)
Harry Potter and the Chamber of Secrets (similarity: 0.96)
Harry Potter and the Deathly Hallows (similarity: 0.96)
Harry Potter and the Half-Blood Prince (similarity: 0.95)
Harry Potter and the Order of the Phoenix (Harry Potter, #5, Part 1) (similarity: 0.86)
The Eternity Code (similarity: 0.84)
James Potter and the Hall of Elders' Crossing (similarity: 0.84)
Harry Potter and the Methods of Rationality (similarity: 0.83)


In [123]:
#Example - not part of a series
book = "To Kill a Mockingbird"
recs = recommend_combined_tf(book, top_k=5)
print(f"Since you have chosen {book} here are some recommendations you may like...")
for title, score in recs:
    print(f"{title} (similarity: {score:.2f})")

Since you have chosen To Kill a Mockingbird here are some recommendations you may like...
Go Set a Watchman (similarity: 0.89)
The Adventures of Huckleberry Finn (similarity: 0.85)
The Adventures of Tom Sawyer (similarity: 0.85)
Paper Moon (similarity: 0.85)
Until Friday Night (similarity: 0.84)


##### Lets analyse the above result. For simplicity we are only considering 4 fields for comparison.

In [130]:
print(books.loc[books['title']=='Go Set a Watchman',['author','genres','characters','setting']])

          author                                             genres  \
2757  Harper Lee  ['Fiction', 'Historical Fiction', 'Classics', ...   

                                             characters  \
2757  ['Scout Finch', 'Atticus Finch', 'Aunt Alexand...   

                                   setting  
2757  ['Maycomb, Alabama (United States)']  


In [132]:
print(books.loc[books['title']=='To Kill a Mockingbird',['author','genres','characters','setting']])

       author                                             genres  \
2  Harper Lee  ['Classics', 'Fiction', 'Historical Fiction', ...   

                                          characters  \
2  ['Scout Finch', 'Atticus Finch', 'Jem Finch', ...   

                                setting  
2  ['Maycomb, Alabama (United States)']  


Both the books 'To kill a MockingBird' and 'Go set a watchman' are written by the same author, the genre is almost same and setting and charachters are similar.

In [135]:
print(books.loc[books['title']=='The Adventures of Huckleberry Finn',['author','genres','characters','setting']])

                                               author  \
47  Mark Twain, Guy Cardwell (Notes), John Seelye ...   

                                               genres  \
47  ['Classics', 'Fiction', 'Historical Fiction', ...   

                                           characters  \
47  ['Huckleberry Finn', 'Tom Sawyer', 'Jim Upton'...   

                                              setting  
47  ['United States of America', 'St. Petersburg, ...  


Wrt "To kill a Mockingbird", the genre is the same. The same goes for "The Adventures of Tom Sawyer" which is written by the same author as "Adventures of Huckleberry Finn"

In [138]:
print(books.loc[books['title']=='Paper Moon',['author','genres','characters','setting']])

                author                                             genres  \
23878  Joe David Brown  ['Fiction', 'Historical Fiction', 'Classics', ...   

      characters                      setting  
23878         []  ['Alabama (United States)']  


"Paper Moon" has the same genre and similar setting 

In [141]:
print(books.loc[books['title']=='Until Friday Night',['author','genres','characters','setting']])

                              author  \
6836  Abbi Glines (Goodreads Author)   

                                                 genres  \
6836  ['Romance', 'Young Adult', 'Contemporary', 'Ne...   

                             characters                              setting  
6836  ['Maggie Carleton', 'West Ashby']  ['Lawton, Alabama (United States)']  


"Until Friday Night" has a similar setting. 